# ee.ipynb — Sliding Window 기반 싸움/스토킹 통합 판정

이 노트북은 기존 파이프라인의 약점이었던 **영상 초반 평균에 의한 점수 희석** 문제를 줄이기 위해, 영상 전체를 2~4초 단위의 sliding window로 나누어 분석합니다.

핵심 변경점은 다음과 같습니다.

- 싸움과 스토킹 모두 **사람 2명 이상의 pair box**를 만들고, pair 단위로 특징을 계산합니다.
- 이상행동이 영상 시작부터 나오지 않아도 잡히도록, 전체 평균 대신 **가장 의심도가 높은 window**를 최종 판단에 반영합니다.
- 싸움은 `근접 + 급격한 움직임 + 접촉/상호작용`을 강하게 봅니다.
- 스토킹은 `일정 거리 유지 + 같은 방향 이동 + 장시간 추적 + 비접촉`을 강하게 봅니다.
- 마지막 결과는 `pandas` 표로 깔끔하게 출력하고, CSV/JSON으로 저장합니다.

In [1]:
# ============================================================
# Cell 1. 필요 라이브러리 안내
# ============================================================
# 아래 라이브러리가 없다면 CMD 또는 Anaconda Prompt에서 먼저 설치하세요.
# pip install ultralytics opencv-python pandas tqdm numpy

import sys
print("Python:", sys.version)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [2]:
# ============================================================
# Cell 2. import
# ============================================================
import json
import math
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = None

try:
    from ultralytics import YOLO
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "ultralytics가 설치되어 있지 않습니다. CMD에서 `pip install ultralytics` 실행 후 다시 시도하세요."
    ) from e

In [3]:
# ============================================================
# Cell 3. 경로 설정
# ============================================================
# 사용자가 지금까지 사용한 구조를 기본 후보로 넣었습니다.
# 본인 PC에서 다른 위치라면 VIDEO_ROOT만 직접 수정하면 됩니다.

VIDEO_ROOT_CANDIDATES = [
    Path(r"C:\Users\우성한\Sample (1)\Sample"),
    Path(r"C:\Users\우성한\Sample\Sample"),
    Path(r"C:\Users\우성한\Sample (1)"),
    Path(r"C:\Users\우성한\Sample"),
    Path("./Sample"),
    Path("./sample"),
    Path("./videos"),
    Path("."),
]

VIDEO_ROOT = next((p for p in VIDEO_ROOT_CANDIDATES if p.exists()), VIDEO_ROOT_CANDIDATES[0])

# 결과 저장 위치
if Path(r"C:\Users\우성한").exists():
    OUTPUT_DIR = Path(r"C:\Users\우성한\outputs\prediction\ee")
else:
    OUTPUT_DIR = Path("./outputs/prediction/ee")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PER_VIDEO_DIR = OUTPUT_DIR / "per_video_json"
PER_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("VIDEO_ROOT:", VIDEO_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("VIDEO_ROOT exists:", VIDEO_ROOT.exists())

VIDEO_ROOT: C:\Users\우성한\Sample (1)\Sample
OUTPUT_DIR: C:\Users\우성한\outputs\prediction\ee
VIDEO_ROOT exists: True


In [4]:
# ============================================================
# Cell 4. 주요 설정값
# ============================================================
# 속도가 너무 느리면 FRAME_STRIDE를 8~10으로 올리세요.
# 더 정확하게 보고 싶으면 FRAME_STRIDE를 3~5로 낮추세요.

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}

YOLO_MODEL_CANDIDATES = ["yolo11n.pt", "yolov8n.pt"]
IMG_SIZE = 640
PERSON_CONF = 0.25

FRAME_STRIDE = 5          # N프레임마다 분석
MAX_MISSING = 12          # track 유지 허용 프레임 수, 분석 프레임 기준
IOU_MATCH_TH = 0.20       # bbox tracker IoU 매칭 기준
MIN_TRACK_HITS = 2        # pair 분석에 사용할 최소 검출 누적 횟수

WINDOW_SEC = 4.0          # sliding window 길이
STEP_SEC = 1.0            # sliding window 이동 간격
MIN_WINDOW_PAIR_FRAMES = 3

# 거리 기준: 사람 중심 거리 / 평균 bbox 높이
CLOSE_DIST_NORM = 1.20
VERY_CLOSE_DIST_NORM = 0.65
STALKING_MIN_DIST_NORM = 1.00
STALKING_MAX_DIST_NORM = 4.50

# 최종 판정 threshold
FIGHT_TH = 0.56
STALKING_TH = 0.58

# Optical flow가 너무 크거나 작은 경우를 막기 위한 정규화 clamp
FLOW_CLIP = 2.50
MOTION_CLIP = 2.50

print("설정 완료")

설정 완료


In [5]:
# ============================================================
# Cell 5. 공통 유틸 함수
# ============================================================
def list_video_files(root: Path) -> List[Path]:
    root = Path(root)
    if root.is_file() and root.suffix.lower() in VIDEO_EXTS:
        return [root]
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.suffix.lower() in VIDEO_EXTS])


def xyxy_to_center(b: List[float]) -> Tuple[float, float]:
    x1, y1, x2, y2 = b
    return (float((x1 + x2) / 2), float((y1 + y2) / 2))


def bbox_wh(b: List[float]) -> Tuple[float, float]:
    x1, y1, x2, y2 = b
    return max(1.0, float(x2 - x1)), max(1.0, float(y2 - y1))


def bbox_area(b: List[float]) -> float:
    w, h = bbox_wh(b)
    return float(w * h)


def bbox_iou(a: List[float], b: List[float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return float(inter / union) if union > 0 else 0.0


def union_bbox(a: List[float], b: List[float], pad: float = 0.10) -> List[float]:
    x1 = min(a[0], b[0])
    y1 = min(a[1], b[1])
    x2 = max(a[2], b[2])
    y2 = max(a[3], b[3])
    w = x2 - x1
    h = y2 - y1
    return [x1 - w * pad, y1 - h * pad, x2 + w * pad, y2 + h * pad]


def clip_bbox(b: List[float], width: int, height: int) -> List[int]:
    x1, y1, x2, y2 = b
    x1 = int(max(0, min(width - 1, round(x1))))
    y1 = int(max(0, min(height - 1, round(y1))))
    x2 = int(max(0, min(width, round(x2))))
    y2 = int(max(0, min(height, round(y2))))
    if x2 <= x1:
        x2 = min(width, x1 + 1)
    if y2 <= y1:
        y2 = min(height, y1 + 1)
    return [x1, y1, x2, y2]


def safe_div(a: float, b: float, default: float = 0.0) -> float:
    return float(a / b) if abs(b) > 1e-9 else float(default)


def norm01(x: float, lo: float, hi: float) -> float:
    if hi <= lo:
        return 0.0
    return float(np.clip((x - lo) / (hi - lo), 0.0, 1.0))


def cosine_sim(v1: Tuple[float, float], v2: Tuple[float, float]) -> float:
    x1, y1 = v1
    x2, y2 = v2
    n1 = math.sqrt(x1 * x1 + y1 * y1)
    n2 = math.sqrt(x2 * x2 + y2 * y2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    return float((x1 * x2 + y1 * y2) / (n1 * n2))


def to_py(obj: Any) -> Any:
    """json 저장을 위해 numpy 타입을 Python 기본 타입으로 변환"""
    if isinstance(obj, dict):
        return {str(k): to_py(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_py(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_py(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj


print("유틸 함수 준비 완료")

유틸 함수 준비 완료


In [6]:
# ============================================================
# Cell 6. YOLO person detector 로드
# ============================================================
def load_yolo_model():
    last_err = None
    for name in YOLO_MODEL_CANDIDATES:
        try:
            model = YOLO(name)
            print("YOLO model loaded:", name)
            return model, name
        except Exception as e:
            last_err = e
            print(f"모델 로드 실패: {name} -> {e}")
    raise RuntimeError(f"YOLO 모델 로드 실패: {last_err}")


PERSON_MODEL, PERSON_MODEL_NAME = load_yolo_model()


def detect_persons(frame: np.ndarray) -> List[Dict[str, Any]]:
    """YOLO로 person class만 추출"""
    results = PERSON_MODEL.predict(
        frame,
        imgsz=IMG_SIZE,
        conf=PERSON_CONF,
        verbose=False,
        classes=[0],
    )
    res = results[0]
    detections = []
    if res.boxes is None:
        return detections

    for box in res.boxes:
        cls = int(box.cls.item()) if box.cls is not None else -1
        if cls != 0:
            continue
        conf = float(box.conf.item()) if box.conf is not None else 0.0
        xyxy = box.xyxy[0].detach().cpu().numpy().astype(float).tolist()
        detections.append({"bbox": xyxy, "conf": conf})
    return detections

YOLO model loaded: yolo11n.pt


In [7]:
# ============================================================
# Cell 7. 간단한 IoU 기반 person tracker
# ============================================================
@dataclass
class Track:
    tid: int
    bbox: List[float]
    conf: float
    first_frame: int
    last_frame: int
    hits: int = 1
    missing: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

    def update(self, bbox: List[float], conf: float, frame_idx: int, t: float):
        prev_center = self.center()
        new_center = xyxy_to_center(bbox)
        self.bbox = bbox
        self.conf = float(conf)
        self.last_frame = int(frame_idx)
        self.hits += 1
        self.missing = 0
        self.history.append({
            "frame_idx": int(frame_idx),
            "t": float(t),
            "bbox": [float(v) for v in bbox],
            "center": [float(new_center[0]), float(new_center[1])],
            "prev_center": [float(prev_center[0]), float(prev_center[1])] if prev_center is not None else None,
            "conf": float(conf),
        })

    def mark_missing(self):
        self.missing += 1

    def center(self) -> Tuple[float, float]:
        return xyxy_to_center(self.bbox)

    def velocity(self) -> Tuple[float, float]:
        if len(self.history) < 2:
            return (0.0, 0.0)
        c2 = self.history[-1]["center"]
        c1 = self.history[-2]["center"]
        return (float(c2[0] - c1[0]), float(c2[1] - c1[1]))


class SimpleIoUTracker:
    def __init__(self, iou_th: float = IOU_MATCH_TH, max_missing: int = MAX_MISSING):
        self.iou_th = iou_th
        self.max_missing = max_missing
        self.next_id = 1
        self.tracks: Dict[int, Track] = {}

    def update(self, detections: List[Dict[str, Any]], frame_idx: int, t: float) -> List[Track]:
        active_ids = list(self.tracks.keys())
        unmatched_tracks = set(active_ids)
        unmatched_dets = set(range(len(detections)))

        # 모든 track-detection IoU를 계산한 뒤 큰 순서로 greedy matching
        candidates = []
        for tid in active_ids:
            for di, det in enumerate(detections):
                candidates.append((bbox_iou(self.tracks[tid].bbox, det["bbox"]), tid, di))
        candidates.sort(reverse=True, key=lambda x: x[0])

        for iou, tid, di in candidates:
            if iou < self.iou_th:
                break
            if tid not in unmatched_tracks or di not in unmatched_dets:
                continue
            det = detections[di]
            self.tracks[tid].update(det["bbox"], det["conf"], frame_idx, t)
            unmatched_tracks.remove(tid)
            unmatched_dets.remove(di)

        # 매칭 안 된 track은 missing 처리
        for tid in list(unmatched_tracks):
            self.tracks[tid].mark_missing()
            if self.tracks[tid].missing > self.max_missing:
                del self.tracks[tid]

        # 매칭 안 된 detection은 새 track 생성
        for di in unmatched_dets:
            det = detections[di]
            tid = self.next_id
            self.next_id += 1
            tr = Track(
                tid=tid,
                bbox=det["bbox"],
                conf=det["conf"],
                first_frame=frame_idx,
                last_frame=frame_idx,
            )
            tr.history.append({
                "frame_idx": int(frame_idx),
                "t": float(t),
                "bbox": [float(v) for v in det["bbox"]],
                "center": [float(xyxy_to_center(det["bbox"])[0]), float(xyxy_to_center(det["bbox"])[1])],
                "prev_center": None,
                "conf": float(det["conf"]),
            })
            self.tracks[tid] = tr

        # 현재 프레임에서 실제 검출된 track만 반환
        present = [tr for tr in self.tracks.values() if tr.last_frame == frame_idx and tr.hits >= MIN_TRACK_HITS]
        return present

In [8]:
# ============================================================
# Cell 8. pair feature 추출
# ============================================================
def roi_flow_score(flow: Optional[np.ndarray], ub: List[float], width: int, height: int, avg_h: float) -> float:
    if flow is None:
        return 0.0
    x1, y1, x2, y2 = clip_bbox(ub, width, height)
    roi = flow[y1:y2, x1:x2]
    if roi.size == 0:
        return 0.0
    mag = np.sqrt(roi[..., 0] ** 2 + roi[..., 1] ** 2)
    # 상위 분위수를 쓰면 작은 움직임보다 갑작스러운 움직임에 더 민감함
    val = float(np.percentile(mag, 85))
    return float(np.clip(safe_div(val, avg_h, 0.0), 0.0, FLOW_CLIP))


def pair_feature_row(
    tr_a: Track,
    tr_b: Track,
    frame_idx: int,
    t: float,
    frame_w: int,
    frame_h: int,
    flow: Optional[np.ndarray],
) -> Dict[str, Any]:
    ba, bb = tr_a.bbox, tr_b.bbox
    ca, cb = tr_a.center(), tr_b.center()
    wa, ha = bbox_wh(ba)
    wb, hb = bbox_wh(bb)
    avg_h = max(1.0, (ha + hb) / 2.0)

    dx = cb[0] - ca[0]
    dy = cb[1] - ca[1]
    dist = math.sqrt(dx * dx + dy * dy)
    dist_norm = safe_div(dist, avg_h, default=999.0)

    va = tr_a.velocity()
    vb = tr_b.velocity()
    speed_a = math.sqrt(va[0] * va[0] + va[1] * va[1]) / avg_h
    speed_b = math.sqrt(vb[0] * vb[0] + vb[1] * vb[1]) / avg_h
    pair_motion = float(np.clip(speed_a + speed_b, 0.0, MOTION_CLIP))
    relative_motion = float(np.clip(math.sqrt((va[0] - vb[0]) ** 2 + (va[1] - vb[1]) ** 2) / avg_h, 0.0, MOTION_CLIP))
    direction_cos = cosine_sim(va, vb)

    # 이전 거리 대비 가까워졌는지 계산
    approach_norm = 0.0
    if len(tr_a.history) >= 2 and len(tr_b.history) >= 2:
        pa = tr_a.history[-2]["center"]
        pb = tr_b.history[-2]["center"]
        prev_dist = math.sqrt((pb[0] - pa[0]) ** 2 + (pb[1] - pa[1]) ** 2)
        approach_norm = np.clip((prev_dist - dist) / avg_h, -MOTION_CLIP, MOTION_CLIP)

    iou = bbox_iou(ba, bb)
    ub = union_bbox(ba, bb, pad=0.15)
    flow_s = roi_flow_score(flow, ub, frame_w, frame_h, avg_h)

    return {
        "frame_idx": int(frame_idx),
        "t": float(t),
        "track_a": int(min(tr_a.tid, tr_b.tid)),
        "track_b": int(max(tr_a.tid, tr_b.tid)),
        "pair_key": f"{min(tr_a.tid, tr_b.tid)}-{max(tr_a.tid, tr_b.tid)}",
        "dist_norm": float(dist_norm),
        "close": bool(dist_norm < CLOSE_DIST_NORM),
        "very_close": bool(dist_norm < VERY_CLOSE_DIST_NORM),
        "moderate_dist": bool(STALKING_MIN_DIST_NORM <= dist_norm <= STALKING_MAX_DIST_NORM),
        "iou": float(iou),
        "speed_a": float(np.clip(speed_a, 0.0, MOTION_CLIP)),
        "speed_b": float(np.clip(speed_b, 0.0, MOTION_CLIP)),
        "pair_motion": float(pair_motion),
        "relative_motion": float(relative_motion),
        "direction_cos": float(direction_cos),
        "approach_norm": float(approach_norm),
        "roi_flow": float(flow_s),
        "union_bbox": [float(v) for v in ub],
        "bbox_a": [float(v) for v in ba],
        "bbox_b": [float(v) for v in bb],
    }

In [9]:
# ============================================================
# Cell 9. window별 fight/stalking score 계산
# ============================================================
def score_pair_window(g: pd.DataFrame) -> Dict[str, Any]:
    """하나의 window 안에서 특정 pair에 대한 점수 계산"""
    n = len(g)
    if n == 0:
        return {}

    close_ratio = float(g["close"].mean())
    very_close_ratio = float(g["very_close"].mean())
    moderate_ratio = float(g["moderate_dist"].mean())
    overlap_ratio = float((g["iou"] > 0.01).mean())

    p90_pair_motion = float(np.percentile(g["pair_motion"], 90))
    p90_relative_motion = float(np.percentile(g["relative_motion"], 90))
    p90_flow = float(np.percentile(g["roi_flow"], 90))
    positive_approach = float(np.maximum(g["approach_norm"].to_numpy(), 0).mean())

    high_motion = ((g["pair_motion"] > 0.20) | (g["relative_motion"] > 0.18) | (g["roi_flow"] > 0.045))
    burst_ratio = float(high_motion.mean())

    same_direction_ratio = float((g["direction_cos"] > 0.45).mean())
    not_close_ratio = float((~g["very_close"]).mean())

    # 싸움: 가까움 + 순간적인 움직임 + ROI flow + 상호작용
    close_score = max(close_ratio, very_close_ratio * 1.15)
    motion_score = max(
        norm01(p90_pair_motion, 0.05, 0.45),
        norm01(p90_relative_motion, 0.04, 0.35),
        norm01(p90_flow, 0.015, 0.12),
    )
    burst_score = norm01(burst_ratio, 0.10, 0.70)
    overlap_score = max(overlap_ratio, norm01(float(g["iou"].max()), 0.00, 0.08))
    approach_score = norm01(positive_approach, 0.01, 0.25)

    fight_score = (
        0.34 * close_score +
        0.32 * motion_score +
        0.16 * burst_score +
        0.10 * overlap_score +
        0.08 * approach_score
    )
    # 근접성이 거의 없으면 싸움으로 보지 않음
    if close_ratio < 0.20 and very_close_ratio < 0.10:
        fight_score *= 0.45

    # 스토킹: 접촉/격한 움직임보다는 일정 거리 + 같은 방향 + 지속성
    duration_score = norm01(n, MIN_WINDOW_PAIR_FRAMES, max(MIN_WINDOW_PAIR_FRAMES + 4, int(WINDOW_SEC * 2)))
    distance_stable = 1.0 - norm01(float(g["dist_norm"].std(ddof=0)), 0.50, 2.50)
    low_burst = 1.0 - norm01(burst_ratio, 0.25, 0.85)
    low_contact = 1.0 - max(very_close_ratio, overlap_ratio)

    stalking_score = (
        0.28 * duration_score +
        0.27 * moderate_ratio +
        0.22 * same_direction_ratio +
        0.13 * distance_stable +
        0.10 * low_contact
    )
    # 싸움 신호가 강하면 스토킹을 약화
    if fight_score >= 0.50 or very_close_ratio > 0.35:
        stalking_score *= 0.55
    # 움직임이 거의 없으면 스토킹도 약화
    if float(g["pair_motion"].mean()) < 0.025 and float(g["roi_flow"].mean()) < 0.010:
        stalking_score *= 0.60

    return {
        "pair_key": str(g["pair_key"].iloc[0]),
        "frame_count": int(n),
        "start_t": float(g["t"].min()),
        "end_t": float(g["t"].max()),
        "fight_score": float(np.clip(fight_score, 0, 1)),
        "stalking_score": float(np.clip(stalking_score, 0, 1)),
        "close_ratio": close_ratio,
        "very_close_ratio": very_close_ratio,
        "moderate_ratio": moderate_ratio,
        "same_direction_ratio": same_direction_ratio,
        "burst_ratio": burst_ratio,
        "p90_pair_motion": p90_pair_motion,
        "p90_relative_motion": p90_relative_motion,
        "p90_flow": p90_flow,
        "mean_dist_norm": float(g["dist_norm"].mean()),
        "std_dist_norm": float(g["dist_norm"].std(ddof=0)),
        "max_iou": float(g["iou"].max()),
    }


def sliding_window_scores(pair_df: pd.DataFrame, duration: float) -> pd.DataFrame:
    if pair_df.empty or duration <= 0:
        return pd.DataFrame()

    rows = []
    starts = np.arange(0, max(0.0, duration - WINDOW_SEC) + STEP_SEC, STEP_SEC)
    if len(starts) == 0:
        starts = np.array([0.0])

    for ws in starts:
        we = min(float(duration), float(ws + WINDOW_SEC))
        wdf = pair_df[(pair_df["t"] >= ws) & (pair_df["t"] < we)].copy()
        if len(wdf) < MIN_WINDOW_PAIR_FRAMES:
            continue
        for _, g in wdf.groupby("pair_key"):
            if len(g) < MIN_WINDOW_PAIR_FRAMES:
                continue
            s = score_pair_window(g)
            if not s:
                continue
            s["window_start"] = float(ws)
            s["window_end"] = float(we)
            rows.append(s)

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)


def summarize_windows(window_df: pd.DataFrame) -> Dict[str, Any]:
    if window_df.empty:
        return {
            "best_fight_score": 0.0,
            "best_stalking_score": 0.0,
            "best_fight_window": None,
            "best_stalking_window": None,
            "final_label": "other",
            "final_score": 0.0,
            "final_reason": "person pair window가 충분히 형성되지 않음",
        }

    best_fight = window_df.sort_values("fight_score", ascending=False).iloc[0].to_dict()
    best_stalk = window_df.sort_values("stalking_score", ascending=False).iloc[0].to_dict()

    fight_score = float(best_fight["fight_score"])
    stalk_score = float(best_stalk["stalking_score"])

    # 최종 분류: 싸움은 근접/급격 움직임이 강할 때 우선.
    if fight_score >= FIGHT_TH and fight_score >= stalk_score * 0.92:
        final_label = "fight"
        final_score = fight_score
        final_reason = "근접한 사람 pair에서 급격한 움직임/flow/접촉 신호가 가장 강함"
        final_window = best_fight
    elif stalk_score >= STALKING_TH:
        final_label = "stalking"
        final_score = stalk_score
        final_reason = "사람 pair가 일정 거리에서 같은 방향으로 지속 이동하는 신호가 강함"
        final_window = best_stalk
    elif fight_score >= FIGHT_TH:
        final_label = "fight"
        final_score = fight_score
        final_reason = "stalking 점수도 있으나 근접 상호작용 신호가 threshold 이상"
        final_window = best_fight
    else:
        final_label = "other"
        final_score = max(fight_score, stalk_score)
        final_reason = "fight/stalking threshold를 넘지 못함"
        final_window = best_fight if fight_score >= stalk_score else best_stalk

    return {
        "best_fight_score": fight_score,
        "best_stalking_score": stalk_score,
        "best_fight_window": to_py(best_fight),
        "best_stalking_window": to_py(best_stalk),
        "final_label": final_label,
        "final_score": float(final_score),
        "final_reason": final_reason,
        "final_window": to_py(final_window),
    }

In [10]:
# ============================================================
# Cell 10. 영상 1개 분석 함수
# ============================================================
def analyze_video(video_path: Path) -> Dict[str, Any]:
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    duration = safe_div(total_frames, fps, default=0.0)

    tracker = SimpleIoUTracker()
    pair_rows = []
    frame_summaries = []

    prev_gray = None
    processed_frames = 0
    max_people_seen = 0

    frame_idx = 0
    pbar_total = total_frames if total_frames > 0 else None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_STRIDE != 0:
            frame_idx += 1
            continue

        t = safe_div(frame_idx, fps, default=0.0)
        detections = detect_persons(frame)
        present_tracks = tracker.update(detections, frame_idx, t)
        max_people_seen = max(max_people_seen, len(present_tracks))

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        flow = None
        if prev_gray is not None and prev_gray.shape == gray.shape:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray,
                gray,
                None,
                pyr_scale=0.5,
                levels=3,
                winsize=15,
                iterations=3,
                poly_n=5,
                poly_sigma=1.2,
                flags=0,
            )
        prev_gray = gray

        # 사람 2명 이상이면 pair box를 새로 구성하여 pair feature 계산
        if len(present_tracks) >= 2:
            for i in range(len(present_tracks)):
                for j in range(i + 1, len(present_tracks)):
                    row = pair_feature_row(
                        present_tracks[i],
                        present_tracks[j],
                        frame_idx=frame_idx,
                        t=t,
                        frame_w=frame_w,
                        frame_h=frame_h,
                        flow=flow,
                    )
                    pair_rows.append(row)

        frame_summaries.append({
            "frame_idx": int(frame_idx),
            "t": float(t),
            "person_detections": int(len(detections)),
            "active_tracks": int(len(present_tracks)),
            "pair_count": int(max(0, len(present_tracks) * (len(present_tracks) - 1) // 2)),
        })

        processed_frames += 1
        frame_idx += 1

    cap.release()

    pair_df = pd.DataFrame(pair_rows)
    window_df = sliding_window_scores(pair_df, duration)
    win_summary = summarize_windows(window_df)

    summary = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "fps": float(fps),
        "total_frames": int(total_frames),
        "duration_sec": float(duration),
        "processed_frames": int(processed_frames),
        "max_people_seen": int(max_people_seen),
        "pair_feature_rows": int(len(pair_rows)),
        "window_rows": int(len(window_df)),
        "final_label": win_summary["final_label"],
        "final_score": round(float(win_summary["final_score"]), 4),
        "best_fight_score": round(float(win_summary["best_fight_score"]), 4),
        "best_stalking_score": round(float(win_summary["best_stalking_score"]), 4),
        "final_reason": win_summary["final_reason"],
    }

    if win_summary.get("final_window") is not None:
        fw = win_summary["final_window"]
        summary.update({
            "event_start_sec": round(float(fw.get("window_start", fw.get("start_t", 0.0))), 2),
            "event_end_sec": round(float(fw.get("window_end", fw.get("end_t", 0.0))), 2),
            "event_pair": fw.get("pair_key", None),
        })
    else:
        summary.update({
            "event_start_sec": None,
            "event_end_sec": None,
            "event_pair": None,
        })

    result = {
        "task": "ee_sliding_window_pair_fight_stalking",
        "settings": {
            "model": PERSON_MODEL_NAME,
            "frame_stride": FRAME_STRIDE,
            "window_sec": WINDOW_SEC,
            "step_sec": STEP_SEC,
            "fight_th": FIGHT_TH,
            "stalking_th": STALKING_TH,
            "close_dist_norm": CLOSE_DIST_NORM,
            "very_close_dist_norm": VERY_CLOSE_DIST_NORM,
        },
        "summary": to_py(summary),
        "best_windows": to_py(win_summary),
        "frame_summaries": to_py(frame_summaries),
        # 용량이 너무 커지는 것을 막기 위해 pair_rows는 그대로 저장하되 필요 없으면 아래 줄을 주석 처리하세요.
        "pair_features": to_py(pair_rows),
        "window_scores": to_py(window_df.sort_values(["fight_score", "stalking_score"], ascending=False).head(50).to_dict("records")) if not window_df.empty else [],
    }
    return result

In [11]:
# ============================================================
# Cell 11. 전체 영상 일괄 처리
# ============================================================
videos = list_video_files(VIDEO_ROOT)
print("찾은 영상 수:", len(videos))
for p in videos[:10]:
    print(" -", p)
if len(videos) > 10:
    print(" ...")

all_rows = []
failed = []

for video_path in tqdm(videos, desc="ee batch analysis"):
    try:
        result = analyze_video(video_path)
        summary = result["summary"]
        all_rows.append(summary)

        out_json = PER_VIDEO_DIR / f"{video_path.stem}_ee.json"
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

    except Exception as e:
        failed.append({
            "video_name": video_path.name,
            "video_path": str(video_path),
            "error": str(e),
            "traceback": traceback.format_exc(limit=2),
        })
        print("실패:", video_path.name, "->", e)

summary_df = pd.DataFrame(all_rows)
failed_df = pd.DataFrame(failed)

summary_path_json = OUTPUT_DIR / "ee_batch_summary.json"
summary_path_csv = OUTPUT_DIR / "ee_batch_summary.csv"
failed_path_json = OUTPUT_DIR / "ee_failed.json"

batch_summary = {
    "task": "ee_sliding_window_pair_fight_stalking",
    "video_root": str(VIDEO_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "total_videos": len(videos),
    "success_count": len(all_rows),
    "failed_count": len(failed),
    "label_counts": summary_df["final_label"].value_counts().to_dict() if not summary_df.empty else {},
    "results": to_py(all_rows),
    "failed": to_py(failed),
}

with open(summary_path_json, "w", encoding="utf-8") as f:
    json.dump(batch_summary, f, ensure_ascii=False, indent=2)

if not summary_df.empty:
    summary_df.to_csv(summary_path_csv, index=False, encoding="utf-8-sig")

with open(failed_path_json, "w", encoding="utf-8") as f:
    json.dump(to_py(failed), f, ensure_ascii=False, indent=2)

print("\n저장 완료")
print("summary json:", summary_path_json)
print("summary csv :", summary_path_csv)
print("failed json :", failed_path_json)

찾은 영상 수: 22
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
 - C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
 ...


ee batch analysis:   0%|          | 0/22 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 12. 결과 표 깔끔하게 출력
# ============================================================
if summary_df.empty:
    print("성공 처리된 영상이 없습니다.")
else:
    view_cols = [
        "video_name",
        "final_label",
        "final_score",
        "best_fight_score",
        "best_stalking_score",
        "event_start_sec",
        "event_end_sec",
        "event_pair",
        "max_people_seen",
        "pair_feature_rows",
        "window_rows",
        "final_reason",
    ]
    view_cols = [c for c in view_cols if c in summary_df.columns]

    display_df = summary_df[view_cols].copy()
    display_df = display_df.sort_values(
        by=["final_label", "final_score"],
        ascending=[True, False],
    ).reset_index(drop=True)

    print("=" * 80)
    print("EE 최종 분류 요약")
    print("=" * 80)
    print("총 영상 수:", len(videos))
    print("성공:", len(summary_df))
    print("실패:", len(failed_df))
    print("\nlabel counts")
    print(summary_df["final_label"].value_counts().to_string())
    print("\n")

    if display is not None:
        display(display_df)
    else:
        print(display_df.to_string(index=False))

    print("\nCSV 저장 위치:", summary_path_csv)
    print("JSON 저장 위치:", summary_path_json)

if not failed_df.empty:
    print("\n실패 목록")
    if display is not None:
        display(failed_df[["video_name", "error"]])
    else:
        print(failed_df[["video_name", "error"]].to_string(index=False))

In [ ]:
# ============================================================
# Cell 13. 특정 label만 빠르게 확인하기
# ============================================================
# 예: fight 영상만 확인
# fight_df = summary_df[summary_df["final_label"] == "fight"].sort_values("final_score", ascending=False)
# display(fight_df[["video_name", "final_score", "event_start_sec", "event_end_sec", "final_reason"]])

# 예: stalking 영상만 확인
# stalking_df = summary_df[summary_df["final_label"] == "stalking"].sort_values("final_score", ascending=False)
# display(stalking_df[["video_name", "final_score", "event_start_sec", "event_end_sec", "final_reason"]])